# Best Practice Analyzer Report - guarded workflow
Locally authored adaptation of the workflow in [Microsoft's Best Practice Analyzer Report notebook](https://github.com/microsoft/semantic-link-labs/blob/main/notebooks/Best%20Practice%20Analyzer%20Report.ipynb), using the public Semantic Link Labs APIs. This is not a read-only diagnostic notebook.

## What this workflow writes
1. A bulk scan analyzes **every eligible semantic model in the explicitly selected workspaces** and appends results to `modelbparesults` in the notebook's attached default lakehouse (`dbo` for schema-enabled lakehouses). It is not limited to one report/model. Repeated scans append history and consume capacity.
2. The model-creation API creates or **overwrites** the `ModelBPA` Direct Lake semantic model in the destination lakehouse workspace.
3. The report-creation API creates or **updates** the `ModelBPA` Power BI report in that workspace.

All three operations are disabled by default. The original notebook's all-accessible-workspaces example is intentionally omitted. Prefer a dedicated diagnostics lakehouse/workspace; confirm that any existing `ModelBPA` items may be replaced before enabling creation. Source models are analyzed, not repaired.

## Prerequisites and run order
Use a Microsoft Fabric notebook with an attached default lakehouse, active supported capacity/Direct Lake, XMLA access, and appropriate permissions to read source models and write destination tables/items. Metadata APIs can require elevated model permissions. If needed, install your approved version of `semantic-link-labs` in a separate cell with `%pip install semantic-link-labs`, or attach an approved Fabric Environment.

Set the source workspace names and destination IDs below. Lakehouse metadata uses the same deployment placeholders as the other notebooks in this folder; render them with the project deployment or attach the intended default lakehouse manually after import. The runtime destination must match the IDs below before any writes are allowed. Run the bulk scan first, inspect errors/output, then enable semantic-model and report creation in order. Separate sessions can reuse existing results/model. After initial setup, normally enable only the bulk scan to append new results. Outputs/results can contain sensitive model metadata and DAX; apply suitable workspace access controls. No cloud operations were run when this notebook was authored.


In [ ]:
SOURCE_WORKSPACES = []  # Explicit workspace names or IDs; never None/all workspaces.
DESTINATION_WORKSPACE_ID = ""
DESTINATION_LAKEHOUSE_ID = ""

RUN_BULK_SCAN = False
RUN_EXTENDED_BPA = False
CREATE_SEMANTIC_MODEL = False
CREATE_REPORT = False
REPLACEMENT_CONFIRMATION = ""  # Set to CREATE OR REPLACE ModelBPA to allow item creation/update.


In [ ]:
def validate_configuration():
    if not any((RUN_BULK_SCAN, CREATE_SEMANTIC_MODEL, CREATE_REPORT)):
        return
    if not DESTINATION_WORKSPACE_ID.strip() or not DESTINATION_LAKEHOUSE_ID.strip():
        raise ValueError("Set the destination workspace and lakehouse IDs before enabling writes.")
    if RUN_BULK_SCAN and (
        not isinstance(SOURCE_WORKSPACES, list)
        or not SOURCE_WORKSPACES
        or any(not isinstance(w, str) or not w.strip() for w in SOURCE_WORKSPACES)
    ):
        raise ValueError("SOURCE_WORKSPACES must be a non-empty list of explicit workspace names or IDs.")
    if (CREATE_SEMANTIC_MODEL or CREATE_REPORT) and REPLACEMENT_CONFIRMATION != "CREATE OR REPLACE ModelBPA":
        raise ValueError("Review the overwrite warning, then set REPLACEMENT_CONFIRMATION to CREATE OR REPLACE ModelBPA.")

def validate_destination(context):
    if str(context.get("defaultLakehouseId", "")).lower() != DESTINATION_LAKEHOUSE_ID.lower():
        raise ValueError("Attached default lakehouse does not match DESTINATION_LAKEHOUSE_ID. Attach the intended lakehouse and restart the session if required.")
    if str(context.get("defaultLakehouseWorkspaceId", "")).lower() != DESTINATION_WORKSPACE_ID.lower():
        raise ValueError("Attached default lakehouse workspace does not match DESTINATION_WORKSPACE_ID.")

validate_configuration()
if any((RUN_BULK_SCAN, CREATE_SEMANTIC_MODEL, CREATE_REPORT)):
    import notebookutils
    import sempy_labs as labs
    import sempy_labs.report as rep

    validate_destination(notebookutils.runtime.context)
    print(f"Confirmed destination lakehouse {DESTINATION_LAKEHOUSE_ID} in workspace {DESTINATION_WORKSPACE_ID}.")
else:
    print("All cloud operations are disabled. Configure explicit targets and enable only the intended operations.")


In [ ]:
if RUN_BULK_SCAN:
    validate_configuration()
    validate_destination(notebookutils.runtime.context)
    labs.run_model_bpa_bulk(
        workspace=SOURCE_WORKSPACES,
        extended=RUN_EXTENDED_BPA,
    )
else:
    print("Bulk scan skipped; no BPA results appended.")


In [ ]:
if CREATE_SEMANTIC_MODEL:
    validate_configuration()
    validate_destination(notebookutils.runtime.context)
    labs.create_model_bpa_semantic_model(
        dataset="ModelBPA",
        lakehouse=DESTINATION_LAKEHOUSE_ID,
        lakehouse_workspace=DESTINATION_WORKSPACE_ID,
    )
else:
    print("Model creation skipped. Existing ModelBPA semantic model, if any, is unchanged by this cell.")


In [ ]:
if CREATE_REPORT:
    validate_configuration()
    validate_destination(notebookutils.runtime.context)
    rep.create_model_bpa_report(
        report="ModelBPA",
        dataset="ModelBPA",
        dataset_workspace=DESTINATION_WORKSPACE_ID,
    )
else:
    print("Report creation/update skipped.")


## Open the report and interpret results
After the enabled operations finish successfully, open the destination workspace and the `ModelBPA` report. The API uses the Direct Lake model over the stored BPA results. Review run history, affected objects, rules, and severity; compare relationship findings with the inventory notebook under `PBI`.

Bulk scanning can log individual model failures and continue; inspect its output and confirm intended model/run coverage in the results table before treating a run as complete. A clean scan does not prove correct business totals, unique relationship keys, correct RLS, or fast visuals. Use the optional missing-row checks and Power BI Performance Analyzer separately.

There is no automatic rollback across lakehouse writes, model creation, and report creation. If a later stage fails, earlier writes may already exist. Investigate before retrying: scanning again appends another run, creating the model overwrites it, and creating the report can update an existing item. This notebook does not suppress errors from the SDK calls.

[Microsoft source workflow](https://github.com/microsoft/semantic-link-labs/blob/main/notebooks/Best%20Practice%20Analyzer%20Report.ipynb) | [Semantic Link Labs APIs](https://semantic-link-labs.readthedocs.io/en/stable/sempy_labs.html) | [Report APIs](https://semantic-link-labs.readthedocs.io/en/stable/sempy_labs.report.html)
